In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor

In [ ]:
df = pd.read_csv("../data/AEP_hourly.csv")

df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.sort_values("Datetime")
df = df.set_index("Datetime")

# Убираем повторяющиеся timestamps
df = df.groupby(level=0)["AEP_MW"].mean().to_frame()

In [3]:
features = [
    "hour",
    "day_of_week",
    "month",
    "day_of_year",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_mean_168",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
]

In [4]:
# Target
y = df["AEP_MW"]

# Features
X = pd.DataFrame(index=df.index)

X["hour"] = df.index.hour
X["day_of_week"] = df.index.dayofweek
X["month"] = df.index.month
X["day_of_year"] = df.index.dayofyear

# Cyclic features
X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)

X["dow_sin"] = np.sin(2 * np.pi * X["day_of_week"] / 7)
X["dow_cos"] = np.cos(2 * np.pi * X["day_of_week"] / 7)

# Lags
X["lag_1"] = y.shift(1)
X["lag_24"] = y.shift(24)
X["lag_168"] = y.shift(168)

X["lag_2"] = y.shift(2)
X["lag_3"] = y.shift(3)
X["lag_6"] = y.shift(6)
X["lag_12"] = y.shift(12)

# Rolling features
X["rolling_mean_24"] = y.shift(1).rolling(24).mean()
X["rolling_mean_168"] = y.shift(1).rolling(168).mean()

# Убираем строки, где ещё нет лагов
data = X.copy()
data["target"] = y
data = data.dropna()

X = data.drop(columns="target")
y_model = data["target"]

print(X.shape)
print(y_model.shape)

(121101, 17)
(121101,)


In [5]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

tscv = TimeSeriesSplit(n_splits=4)

mae_scores = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):

    X_train_cv = X.iloc[train_idx]
    X_test_cv = X.iloc[test_idx]

    y_train_cv = y_model.iloc[train_idx]
    y_test_cv = y_model.iloc[test_idx]

    model_cv = HistGradientBoostingRegressor(
        max_iter=300,
        random_state=42
    )

    model_cv.fit(X_train_cv, y_train_cv)

    pred_cv = model_cv.predict(X_test_cv)

    mae = mean_absolute_error(y_test_cv, pred_cv)
    rmse = mean_squared_error(y_test_cv, pred_cv) ** 0.5

    mae_scores.append(mae)
    rmse_scores.append(rmse)

    print(f"Fold {fold}: MAE = {mae:.2f}, RMSE = {rmse:.2f}")

print()
print(f"Mean MAE: {np.mean(mae_scores):.2f}")
print(f"Mean RMSE: {np.mean(rmse_scores):.2f}")
print(f"MAE std: {np.std(mae_scores):.2f}")
print(f"RMSE std: {np.std(rmse_scores):.2f}")

Fold 1: MAE = 127.90, RMSE = 188.63
Fold 2: MAE = 115.65, RMSE = 153.45
Fold 3: MAE = 120.49, RMSE = 156.89
Fold 4: MAE = 118.24, RMSE = 157.47

Mean MAE: 120.57
Mean RMSE: 164.11
MAE std: 4.57
RMSE std: 14.24


In [6]:
print(f"\nMean MAE: {np.mean(mae_scores):.2f}")
print(f"Mean RMSE: {np.mean(rmse_scores):.2f}")
print(f"MAE std: {np.std(mae_scores):.2f}")
print(f"RMSE std: {np.std(rmse_scores):.2f}")


Mean MAE: 120.57
Mean RMSE: 164.11
MAE std: 4.57
RMSE std: 14.24
